### INITIALIZATION: 
IMPORT TOOLS, SET SEED, AND CREATE INDEPENDENT TABLES

In [1]:
# libraries
import pandas as pd
import numpy as np
import uuid
from datetime import datetime, timedelta

# seed
SEED = 42
np.random.seed(SEED)


print("Libraries loaded and seed set.")

Libraries loaded and seed set.


### Independent Entities

In [2]:
# Step 2: Independent Entities (Departments)
departments_data = [
    {"name": "Traffic & Transport", "daily_capacity": 50, "vulnerability_to_storm": 5.0, "base_rate": 20},
    {"name": "Public Works", "daily_capacity": 40, "vulnerability_to_storm": 8.0, "base_rate": 15},
    {"name": "Parks & Recreation", "daily_capacity": 10, "vulnerability_to_storm": 1.5, "base_rate": 2},
    {"name": "Animal Control", "daily_capacity": 15, "vulnerability_to_storm": 1.0, "base_rate": 5},
    {"name": "Sanitation", "daily_capacity": 60, "vulnerability_to_storm": 3.0, "base_rate": 45}
]

df_departments = pd.DataFrame(departments_data)

# Give each department a unique UUID
df_departments['department_id'] = [str(uuid.uuid4()) for _ in range(len(df_departments))]

print(df_departments.head())

                  name  ...                         department_id
0  Traffic & Transport  ...  c5162128-509d-47ec-a0f8-0ecb85426886
1         Public Works  ...  b44ac955-9af6-4694-bc52-bd767ae6dac0
2   Parks & Recreation  ...  af25914c-de29-46eb-aaf9-22e1cd60ebbb
3       Animal Control  ...  90384748-46ab-4b62-ba78-6ad76e9cb5e6
4           Sanitation  ...  71df378b-810e-4c6e-a2da-7a971a388f2b

[5 rows x 5 columns]


In [3]:
# setup 
NUM_CITIZENS = 10000
BASE_DATE = datetime(2026, 8, 1)

# citizen ids
# generate 10000 ids for each citizen
citizen_ids = [str(uuid.uuid4()) for _ in range(NUM_CITIZENS)]

# dates
# generate random dates
random_days_ago = np.random.randint(1, 365,size=NUM_CITIZENS)
join_dates = [
    (BASE_DATE - timedelta(days=int(days))).strftime("%Y-%m-%d") 
    for days in random_days_ago
]

# clip L_civic
# np.random.normal creates the raw data without clipping 
# np.clip forces any number below 0.1 to become 0.1, and any above 2 to become 2.
raw_scores = np.random.normal(loc=0.5, scale=0.3, size=NUM_CITIZENS)
L_civics = np.clip(raw_scores, a_min=0.1, a_max=2.0)


# creation of data frame
df_citizens = pd.DataFrame({
    "citizen_id": citizen_ids,
    "join_date": join_dates,
    "L_civic": L_civics,
})

print(df_citizens.head())
print("\nL_civic:")
print(df_citizens["L_civic"].describe())

                             citizen_id   join_date   L_civic
0  69b2f55e-d12b-4d61-9b18-f92af13c17ed  2026-04-20  0.798459
1  11861297-f514-412f-a7ca-c93c0cbb0b8a  2025-08-17  0.471951
2  3aed9844-5a94-457d-ac7a-27cd81539989  2025-11-03  1.152113
3  fae48527-13b2-49bc-be52-9580a5a5c861  2026-04-16  0.100000
4  a0d323ac-b7c8-43dc-883a-7ea7b7761206  2026-05-21  0.211590

L_civic:
count    10000.000000
mean         0.511578
std          0.275631
min          0.100000
25%          0.296869
50%          0.499840
75%          0.701194
max          1.525584
Name: L_civic, dtype: float64


### Step 3B: Timeline & Latent Weather Shock
Simulate a 30-day calendar starting from  (August 1, 2026) and inject a hidden omitted variable  representing a decaying typhoon shock.

In [4]:
# Step 3B: Timeline & Latent Weather Shock

# 1. 30 consecutive days starting from BASE_DATE using timedelta list comprehension
timeline_dates = [
    (BASE_DATE + timedelta(days=i)).strftime("%Y-%m-%d")
    for i in range(30)
]

# 2. Vectorized initialization of latent storm shock (L_storm)
L_storm = np.zeros(30)

# 3. Inject decaying typhoon shock at specific indexes (Day 15, Day 16, Day 17)
L_storm[14] = 1.0  # Index 14 (Day 15): Peak typhoon shock
L_storm[15] = 0.6  # Index 15 (Day 16): Receding floodwaters
L_storm[16] = 0.2  # Index 16 (Day 17): Residual shock

# Create timeline DataFrame
df_timeline = pd.DataFrame({
    "date": timeline_dates,
    "L_storm": L_storm
})

# Verify storm pulse injection
print("Timeline created. Storm pulse slice (index 13:18):")
print(df_timeline.iloc[13:18])

Timeline created. Storm pulse slice (index 13:18):
          date  L_storm
13  2026-08-14      0.0
14  2026-08-15      1.0
15  2026-08-16      0.6
16  2026-08-17      0.2
17  2026-08-18      0.0
